# Preprocess yearly Parquet into regular 10-minute rows

Run notebook 02_parquet_setup/01_setup_yearly_parquets.ipynb first to create the source Parquet, then run this notebook. It preserves `Data_Parquets` and writes **Data_Parquets_10min/<year>/Station_<ID>/data.parquet**. Each file has a full year's 10-minute grid: a timestamp identifies `[t, t + 10 minutes)`. Plotting notebooks 04Ã¢â‚¬â€œ08 and model notebook 05_model_training/icing_8230_8234_8235_8236/01_horizon_10min/01_train_all_features.ipynb read this processed format.

Columns added:
- `sensorObserved`: a real sensor row existed; inserted rows are not observations.
- **`selectedAlarmOccurred`**: `True` only if **3130, 8000, 8235 or 10105** started in this interval (the selected training targets).
- `selectedTriggeredAlarmCodes`, `selectedTriggeredAlarmCounts`, `selectedAlarmTriggerCount`: selected codes and counts only.
- `alarmOccurred`: at least one alarm of **any** code started; retained for compatibility and auditing. It is not the selected-target label.
- `triggeredAlarmCodes`, `triggeredAlarmCounts`, `alarmTriggerCount`: starting codes, JSON counts per code, and total starts (repeated events count separately).
- `alarmOn`: `yes` if any recorded alarm overlaps this interval, otherwise `no`.
- `activeAlarmCodes`: codes active at any point in the interval.
- `alarmEvents`: original start/end times as JSON, retained for precise forecasting labels, not as extra timestamp rows or model features.

An alarm ending exactly at the bin boundary does not flag the next bin. Unknown/zero end duration flags the start bin only; invalid negative duration stops preprocessing. Missing sensor values stay missing; no imputation or interpolation is applied. Sensors are expected on the 10-minute grid already; off-grid readings stop for an explicit alignment decision. Duplicate sensor timestamps keep the last row. Available earlier years' alarm files are checked for carry-in events; unrecorded events cannot be inferred. All supplied alarm codes are preserved; plotting filters are applied later.

Unchanged output is reused. Set `REBUILD = True` after changing the preprocessing policy. After changing source data, rerun notebook 02_parquet_setup/01_setup_yearly_parquets.ipynb then this notebook. Train models again after changing datasets; old model reports do not describe the new dataset.

`SELECTED_ALARM_CODES` is editable. Changing it invalidates the preprocessing cache automatically. All nonselected events, codes and counts remain in the original alarm columns. For independent models, use each code's own trigger count/label; the combined selected flag means any of the four started, not that all four did.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'notebooks' / '01_data_export/01_export_yearly_turbine_scada.ipynb').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open from Iteration2 or its notebooks folder.')
if str(ROOT / 'notebooks' / 'shared') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks' / 'shared'))
import importlib
import interval_parquet
importlib.reload(interval_parquet)
from interval_parquet import preprocess_year, load_aligned

SOURCE_ROOT = ROOT / 'Data_Parquets'
OUTPUT_ROOT = ROOT / 'Data_Parquets_10min'
YEARS = None  # None = all prepared years; or [2016, 2017, 2022].
TURBINE_IDS = None  # All turbines; or ['2304511'].
REBUILD = False

SELECTED_ALARM_CODES = {"3130", "8000", "8235", "10105"}


In [ ]:
available = sorted({int(p.name.removeprefix('turbines_')) for p in SOURCE_ROOT.iterdir()
                    if p.is_dir() and p.name.removeprefix('turbines_').isdigit()})
years = available if YEARS is None else list(dict.fromkeys(YEARS))
if not years:
    raise ValueError('No source years available; run notebook 02_parquet_setup/01_setup_yearly_parquets.ipynb first.')
summary = []
for year in years:
    summary.extend(preprocess_year(SOURCE_ROOT, OUTPUT_ROOT, year, TURBINE_IDS, REBUILD,
                                   selected_alarm_codes=SELECTED_ALARM_CODES))
display(pd.DataFrame(summary))
print('Processed data:', OUTPUT_ROOT)


In [ ]:
first = summary[0]
preview, _ = load_aligned(OUTPUT_ROOT, first['Year'], first['Turbine'])
display(preview.loc[preview.selectedAlarmOccurred].head(10))
print('Grid spacing:', preview.index.to_series().diff().dropna().unique())
print('Recorded starts:', int(preview.alarmTriggerCount.sum()))
print('Selected alarm starts:', int(preview.selectedAlarmTriggerCount.sum()))
